# 검색 알고리즘
- 검색 알고리즘은 사용자가 작성한 쿼리와 참조 문서간의 관련성을 평가하고 가장 적합한 문서를 선별하는 로직을 의미한다. 
- 검색 알고리즘은 희소 검색(sparse retrieval), 밀집 검색(dense retrieval) 알고리즘이 있다. 

## 희소 검색(sparse retireval)

- 희소 검색은 문서와 쿼리를 희소 벡터 형태로 표현하여 검색을 수행한다. 
- 희소 벡터는 전체 어휘 사전의 크기에 해당하는 차원을 가진 벡터로 해당 문서나 쿼리에 등장하는 단어에 해당하는 위치만 1이고 나머지는 모두 0인 형태를 가진다. 

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 


def load_file(path: str):
    try:
        return PyPDFLoader(path)
    except FileNotFoundError:
        print("File not found.")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

def create_text_splitter(chunk_size: int, chunk_overlap: int):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap) 


file_path = "./data/투자설명서.pdf"
loader =load_file(file_path)
text_splitter = create_text_splitter(1000, 200)
docs = loader.load_and_split(text_splitter)

/Users/jbd/.pyenv/versions/inflearn-llm-application/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_classic.retrievers import BM25Retriever 
from kiwipiepy import Kiwi 

kiwi_tokenizer = Kiwi()

def kiwi_tokenize(text):
    return [token.form for token in kiwi_tokenizer.tokenize(text)]

In [5]:
bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=kiwi_tokenize)
bm25_retriever.k = 2

In [6]:
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_openai import OpenAI


def call_model(model : str = "google/gemma-4-e4b", temperature: float = 0.5) -> OpenAI:
    return OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model = model,
    temperature=temperature
)
    

qa_chain = RetrievalQA.from_chain_type(
    llm = call_model(),
    retriever = bm25_retriever, 
    chain_type="stuff", 
    return_source_documents=True,
)


In [7]:
qa_chain.invoke("이 회사가 발행한 주식의 총 발행향이 어느 정도야?")

{'query': '이 회사가 발행한 주식의 총 발행향이 어느 정도야?',
 'result': ' 이 회사가 발행한 주식의 총 발행향은 13,602,977주야.',
 'source_documents': [Document(metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': './data/투자설명서.pdf', 'total_pages': 514, 'page': 41, 'page_label': '42'}, page_content='아니하고 청약의 권유를 받은 자를 합산한다. 다만, 다음 각 호의 어느 하나에 해당하는 자는 합산\n대상자에서 제외한다. <개정 2009. 10. 1., 2010. 12. 7., 2013. 6. 21., 2013. 8. 27., 2016. 6. 28.,\n2016. 7. 28.>\n1. 다음 각 목의 어느 하나에 해당하는 전문가\n가. 전문투자자\n나. 삭제<2016. 6. 28.>\n다. 「공인회계사법」에 따른 회계법인\n라. 신용평가회사(법 제335조의3에 따라 신용평가업인가를 받은 자를 말한다. 이하 같다)\n마. 발행인에게 회계, 자문 등의 용역을 제공하고 있는 공인회계사ㆍ감정인ㆍ변호사ㆍ변리사ㆍ세무\n사 등 공인된\n자격증을 가지고 있는 자\n바. 그 밖에 발행인의 재무상황이나 사업내용 등을 잘 알 수 있는 전문가로서 금융위원회가 정하여\n고시하는 자\n2. 다음 각 목의 어느 하나에 해당하는 연고자\n가. 발행인의 최대주주[「금융회사의 지배구조에 관한 법률」 제2조제6호가목에 따른 최대주주를\n말한다. 이 경\n우 "금융회사"는 "법인"으로 보고, "발행주식(출자지분을 포함한다. 이하 같다)"은 "발행주식"으로 본\n다. 이하\n같다]와 발행

## 밀집검색 (dense retrieval)
- 문서와 고차원의 쿼리를 밀집 벡터 형태로 표현하여 검색을 구행하는 방법이다. 
- 희소 검색과 달리 단어이 존재 여부나 빈도만이 아닌 단어의 의미와 문맥을 고려하여 검색을 수행한다. 
- 트랜스포머 기반의 임베딩 모델을 사용하여 텍스트를 의미 공간에 매핑하는 것이다. 

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = load_file(file_path)
doc_splitter = create_text_splitter(2000, 200) 
docs = loader.load_and_split(doc_splitter) 


In [9]:
from langchain_openai.embeddings import OpenAIEmbeddings 

In [10]:
embedding = OpenAIEmbeddings(
            model="bge-m3",
            base_url="http://localhost:1234/v1",
            api_key="lm-studio",
            check_embedding_ctx_length=False)

In [11]:
# from langchain_community.vectorstores import FAISS 
from langchain_qdrant import QdrantVectorStore

qdrant_store = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embedding,
    url="http://localhost:6333",
    collection_name="investment_docs",
)

# faiss_store = FAISS.from_documents(docs, embedding)

In [13]:
#faiss_store.save_local("./content/DB")

In [ ]:
# persist_directory = "./content/DB"
# vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

In [15]:
qdrant_retriever = qdrant_store.as_retriever(search_kwargs={"k":2})
#faiss_retriever = faiss_store.as_retriever(search_kwargs={"k":2})

In [18]:
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_openai import ChatOpenAI 


def call_chat_model(model : str = "google/gemma-4-e4b", temperature: float = 0.5) -> OpenAI:
    return ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model = model,
    temperature=temperature
)

qa_chain = RetrievalQA.from_chain_type(
    llm = call_chat_model(),
    chain_type = "stuff", 
    retriever = qdrant_retriever, 
    return_source_documents=True 
)

In [19]:

result = qa_chain.invoke({
    "query": "이 회사가 발행한 주식의 총 발행량이 어느 정도야?"
})

print(result["result"])

제공된 자료에 따르면, **현재까지 발행한 주식의 총수는 13,602,977주**입니다.

이는 '5. 정관에 관한 사항' 항목의 **Ⅱ. 현재까지 발행한 주식의 총수** 및 **Ⅳ. 발행주식의 총수**에 명시되어 있습니다.
